In [ ]:
from ultralytics import YOLO
import cv2
import torch
import torchvision.transforms as transforms
from PIL import Image
import json
import torch.nn as nn
from torchvision import models

In [ ]:
yolo_model = YOLO("yolov8n.pt")  # lightweight & fast

In [ ]:
# Load class names
with open("../saved_models/classes.json", "r") as f:
    class_names = json.load(f)

# Recreate model
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, len(class_names))

# Load weights
model.load_state_dict(torch.load("../saved_models/animal_classifier.pth"))
model.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
cap = cv2.VideoCapture(0)

animal_classes = [14,15,16,17,18,19,20,21,22,23]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo_model(frame)[0]

    for box in results.boxes:
        cls_id = int(box.cls[0])
        confidence = float(box.conf[0])

        # Filter: only animals
        if cls_id not in animal_classes:
            continue

        # Filter: confidence threshold
        if confidence < 0.5:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        crop = frame[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        img = transform(img).unsqueeze(0)

        with torch.no_grad():
            outputs = model(img)
            _, predicted = torch.max(outputs, 1)
            label = class_names[predicted.item()]

        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

        # Show BOTH labels
        yolo_label = yolo_model.names[cls_id]
        final_label = f"{yolo_label} → {label}"

        cv2.putText(frame, final_label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    # ✅ Show frame ONCE per loop
    cv2.imshow("WildVision", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()